In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import cv2
from tqdm import tqdm
import time
import torch.optim.lr_scheduler as lr_scheduler 

SEED = 42
torch.manual_seed(SEED)

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: ---{DEVICE}----- uwu")

Using: ---cuda----- uwu


In [3]:
k_root = '/kaggle/input/diabetic-retinopathy-resized'
cvs_path = os.path.join(k_root, 'trainLabels.csv') 
img_dir = os.path.join(k_root, 'resized_train', 'resized_train') 
print(f"Pre-training CSV: {cvs_path}")

MODEL_SAVE_PATH = 'output_models/pretrain_weights.pth'
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True) 


Pre-training CSV: /kaggle/input/diabetic-retinopathy-resized/trainLabels.csv


In [4]:
BATCH_SIZE = 32


In [5]:
class retinopathy_dataset(Dataset):

    def __init__(self, csv_file, image_dir, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform
        
    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_id = self.data_frame.loc[idx, 'image'] 
        img_path = os.path.join(self.image_dir, f"{img_id}.jpeg")
        image = cv2.imread(img_path)
        if image is None:
            img_path = os.path.join(self.image_dir, f"{img_id}.jpg")
            image = cv2.imread(img_path)
            if image is None:
                 raise FileNotFoundError(f"Image not found :C {img_path}")
        
        raw_label = self.data_frame.loc[idx, 'level']
        binary_label = 1.0 if raw_label > 0 else 0.0

        if self.transform:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 
            image = self.transform(image)

        return image, torch.tensor(binary_label, dtype=torch.float32)

# Innitializing the MODELLLLL

In [6]:
from torchvision.models import densenet201, DenseNet201_Weights

model = densenet201(weights=DenseNet201_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

Downloading: "https://download.pytorch.org/models/densenet201-c1103571.pth" to /root/.cache/torch/hub/checkpoints/densenet201-c1103571.pth
100%|██████████| 77.4M/77.4M [00:00<00:00, 203MB/s]


In [7]:
num_fetures = model.classifier.in_features

model.classifier = nn.Sequential(
    nn.Linear(num_fetures, 1)
    
)
for param in model.classifier.parameters():
        param.requires_grad = True

In [8]:
model = model.to(DEVICE)

In [9]:
pretrain_transforms = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)), 
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])


In [10]:
pretrain_dataset = retinopathy_dataset(cvs_path, img_dir, transform=pretrain_transforms)

In [11]:
train_size = int(0.9 * len(pretrain_dataset))
val_size = len(pretrain_dataset) - train_size
pretrain_train_dataset, pretrain_val_dataset = torch.utils.data.random_split(
        pretrain_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED)
    )

In [12]:
pretrain_loader = DataLoader(pretrain_dataset, batch_size=32, shuffle=True, num_workers=4) 
train_loader = DataLoader(pretrain_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4) 
val_loader = DataLoader(pretrain_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4) 

In [13]:
    print(f"Pre-training data ready. Total samples: {len(pretrain_dataset)}")
    print(f"  Train Split (90%): {len(pretrain_train_dataset)} samples")
    print(f"  Validation Split (10%): {len(pretrain_val_dataset)} samples")

Pre-training data ready. Total samples: 35126
  Train Split (90%): 31613 samples
  Validation Split (10%): 3513 samples


In [14]:
optimizer = optim.AdamW(model.parameters(), lr=1e-5) 
criterion = nn.BCEWithLogitsLoss()

In [15]:
scheduler = lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min',        
        factor=0.5,        
        patience=2,        
        verbose=True
    )

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


# TRAININGIGIGNGIGNIGNGIGN LOOOPY

In [16]:
start_time = time.time()
TRAIN_EPOCH = 50
BATCH_SIZE = 32
best_loss = float('inf')

In [17]:
for epoch in range(TRAIN_EPOCH):
    model.train()
    train_loss_counter = 0
    for inputs, labels in tqdm(train_loader, desc=f"Pretrain epoch {epoch+1}/{TRAIN_EPOCH}"):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
        optimizer.zero_grad() 
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    train_loss_counter += loss.item() * inputs.size(0)
    avg_train_loss = train_loss_counter / len(pretrain_train_dataset)
    
    model.eval()
    val_loss_counter = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss_counter += loss.item() * inputs.size(0)
    avg_val_loss = val_loss_counter / len(pretrain_val_dataset)
    
    print(f"\nEpoch {epoch+1} complete!! Train Loss of: {avg_train_loss:.4f} Val Loss of: {avg_val_loss:.4f}")
    
    scheduler.step(avg_val_loss)
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f'     ----model has SAVED and finally improved at {epoch+1}')
    else:
        print(f'     ----model has stopped improving from the best loss of {best_loss:.4f}.')
end_time = time.time()
print(f"\nPre-training complete in JSUTUUTTUSUTUTTT!! [{(end_time - start_time) / 60:.2f}] minutes.")

print("PREtraining ISISISISISISSS finished!!!! YIPEPEPEPEEE")
print(f"=== FINAL best weights are are : {MODEL_SAVE_PATH}) ==== AND best loss is: {best_loss:.4f} === ")

Pretrain epoch 1/50: 100%|██████████| 988/988 [03:42<00:00,  4.43it/s]



Epoch 1 complete!! Train Loss of: 0.0005 Val Loss of: 0.5853
     ----model has SAVED and finally improved at 1


Pretrain epoch 2/50: 100%|██████████| 988/988 [03:10<00:00,  5.17it/s]



Epoch 2 complete!! Train Loss of: 0.0005 Val Loss of: 0.5750
     ----model has SAVED and finally improved at 2


Pretrain epoch 3/50: 100%|██████████| 988/988 [03:10<00:00,  5.18it/s]


...


Epoch 48 complete!! Train Loss of: 0.0004 Val Loss of: 0.5126
     ----model has SAVED and finally improved at 48


Pretrain epoch 49/50: 100%|██████████| 988/988 [03:16<00:00,  5.02it/s]



Epoch 49 complete!! Train Loss of: 0.0004 Val Loss of: 0.5123
     ----model has SAVED and finally improved at 49


Pretrain epoch 50/50: 100%|██████████| 988/988 [03:14<00:00,  5.09it/s]



Epoch 50 complete!! Train Loss of: 0.0005 Val Loss of: 0.5125
     ----model has stopped improving from the best loss of 0.5123.

Pre-training complete in JSUTUUTTUSUTUTTT!! [178.89] minutes.
PREtraining ISISISISISISSS finished!!!! YIPEPEPEPEEE
=== FINAL best weights are are : output_models/pretrain_weights.pth) ==== AND best loss is: 0.5123 === 
